# Infra setup

Creates the catalog objects for the target the job runs in (schemas and the raw volume) and gives the job service principal the access it needs. Safe to run on every job run: everything is created only when missing and grants are only issued when they are not there yet.

In [0]:
%run ../config/config

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.workspace import AclPermission

w = WorkspaceClient()
current_user = spark.sql("SELECT current_user()").first()[0]
schemas = [f"{CATALOG}.{s}" for s in (BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA)]
print(f"running as {current_user}, service principal {SERVICE_PRINCIPAL or '-'}, schemas {schemas}")

In [0]:
for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {RAW_VOLUME}")

In [0]:
def granted(principal, securable_type, securable):
    rows = spark.sql(f"SHOW GRANTS `{principal}` ON {securable_type} {securable}").collect()
    return {r.ActionType for r in rows}


if SERVICE_PRINCIPAL and SERVICE_PRINCIPAL != current_user:
    catalog_grants = granted(SERVICE_PRINCIPAL, "CATALOG", CATALOG)
    if "ALL PRIVILEGES" not in catalog_grants:
        missing = {"USE CATALOG", "CREATE SCHEMA"} - catalog_grants
        if missing:
            spark.sql(f"GRANT {', '.join(sorted(missing))} ON CATALOG {CATALOG} TO `{SERVICE_PRINCIPAL}`")
        for schema in schemas:
            if "ALL PRIVILEGES" not in granted(SERVICE_PRINCIPAL, "SCHEMA", schema):
                spark.sql(f"GRANT ALL PRIVILEGES ON SCHEMA {schema} TO `{SERVICE_PRINCIPAL}`")
    print(f"{SERVICE_PRINCIPAL}: catalog grants {sorted(granted(SERVICE_PRINCIPAL, 'CATALOG', CATALOG))}")

In [0]:
if SERVICE_PRINCIPAL and SERVICE_PRINCIPAL != current_user:
    try:
        acl = w.secrets.get_acl(scope=SECRET_SCOPE, principal=SERVICE_PRINCIPAL)
        has_read = acl.permission in (AclPermission.READ, AclPermission.WRITE, AclPermission.MANAGE)
    except NotFound:
        has_read = False
    if not has_read:
        w.secrets.put_acl(scope=SECRET_SCOPE, principal=SERVICE_PRINCIPAL, permission=AclPermission.READ)
    print(f"{SERVICE_PRINCIPAL}: {w.secrets.get_acl(scope=SECRET_SCOPE, principal=SERVICE_PRINCIPAL).permission} on {SECRET_SCOPE}")